In [ ]:
#!pip install librosa tqdm scikit-learn

In [ ]:
MY_CONFIG = {
    "batch_size": 512, # the official setup typically uses a large batch
    "epochs": 50, # the official MIMII baseline uses relatively few epochs
    "input_dim":320,
}

BASE_PATH = "/content/mimii_data/fan"
SELECTED_IDS = ["id_00", "id_02", "id_04", "id_06"]
ZIP_PATH = "/content/drive/MyDrive/0_dB_fan.zip"
LOCAL_DATA_PATH = "/content/mimii_data"
SAVE_DIR = "/content/drive/MyDrive/MIMII_Baseline_Results"


In [ ]:
import os
import sys
import glob
import numpy as np
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from tqdm import tqdm
from sklearn import metrics
import logging

# ==========================================
# 1. Official feature extraction (exactly matches baseline.py)
# ==========================================
def file_to_vector_array(file_name, n_mels=64, frames=5, n_fft=1024, hop_length=512, power=2.0):
    # Load audio
    y, sr = librosa.load(file_name, sr=None, mono=True)

    # 1. Compute the mel spectrogram
    mel_spectrogram = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=power
    )

    # 2. Convert to log-mel (official formula: 20.0 / power * log10)
    log_mel_spectrogram = 20.0 / power * np.log10(mel_spectrogram + sys.float_info.epsilon)

    # 3. Frame concatenation (vector concatenation)
    # Official logic: concatenate `frames` adjacent frames into one long vector
    vectorarray_size = log_mel_spectrogram.shape[1] - frames + 1
    if vectorarray_size < 1:
        return np.empty((0, n_mels * frames), float)

    dims = n_mels * frames
    vectorarray = np.zeros((vectorarray_size, dims), float)
    for t in range(frames):
        vectorarray[:, n_mels * t: n_mels * (t + 1)] = log_mel_spectrogram[:, t: t + vectorarray_size].T

    return vectorarray

# ==========================================
# 2. PyTorch port of the official model (64-64-8-64-64)
# ==========================================
class MIMII_Baseline_AE(nn.Module):
    def __init__(self, input_dim):
        super(MIMII_Baseline_AE, self).__init__()
        # Encoder: 320 -> 64 -> 64 -> 8
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 8),
            nn.ReLU()
        )
        # Decoder: 8 -> 64 -> 64 -> 320
        self.decoder = nn.Sequential(
            nn.Linear(8, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim) # the official MIMII baseline has no activation on the output layer
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out

# ==========================================
# 3. Core training and evaluation logic
# ==========================================
def train_baseline(target_dir, config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- A. Data preparation ---
    # Replicate the official data split
    normal_files = sorted(glob.glob(f"{target_dir}/normal/*.wav"))
    abnormal_files = sorted(glob.glob(f"{target_dir}/abnormal/*.wav"))

    # Training set: a subset of the normal clips
    # Evaluation set: the first part of the normal clips + all abnormal clips
    train_files = normal_files[len(abnormal_files):]
    eval_files = normal_files[:len(abnormal_files)] + abnormal_files
    eval_labels = [0] * len(normal_files[:len(abnormal_files)]) + [1] * len(abnormal_files)

    print(f"Loading features for {target_dir}...")
    all_features = []
    for f in tqdm(train_files):
        vec = file_to_vector_array(f)
        all_features.append(vec)

    train_x = np.concatenate(all_features, axis=0)
    train_x = torch.FloatTensor(train_x)

    # Equivalent of Keras validation_split=0.1
    dataset = TensorDataset(train_x, train_x) # autoencoder: input == target
    val_size = int(len(dataset) * 0.1)
    train_size = len(dataset) - val_size
    train_db, val_db = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_db, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_db, batch_size=config['batch_size'], shuffle=False)

    # --- B. Model initialization ---
    input_dim = MY_CONFIG['input_dim']
    model = MIMII_Baseline_AE(input_dim).to(device)
    optimizer = optim.Adam(model.parameters())
    criterion = nn.MSELoss()

    # --- C. Training ---
    print(f"Training Baseline...")
    for epoch in range(config['epochs']):
        model.train()
        t_loss = 0
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, x)
            loss.backward()
            optimizer.step()
            t_loss += loss.item()

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1} | Loss: {t_loss/len(train_loader):.6f}")

    # --- D. Evaluation (follows the official AUC computation exactly) ---
    print(f"Evaluating...")
    model.eval()
    y_pred = []
    with torch.no_grad():
        for file_name in tqdm(eval_files):
            # Convert each file to a vector array
            data = file_to_vector_array(file_name)
            data_tensor = torch.FloatTensor(data).to(device)

            # Per-frame reconstruction error (MSE)
            reconstructed = model(data_tensor)
            # Official logic: MSE per row (frame) first, then the mean over all frames of the file
            mse_per_frame = torch.mean((data_tensor - reconstructed)**2, dim=1)
            file_error = torch.mean(mse_per_frame).cpu().item()
            y_pred.append(file_error)

    auc = metrics.roc_auc_score(eval_labels, y_pred)

    # Build the output file name
    # Note: DRIVE_SAVE_PATH here would need to be a global or passed in via config
    model_name = f"baseline_fan_{os.path.basename(target_dir)}.pth"
    full_save_path = os.path.join("/content/drive/MyDrive/MIMII_Baseline_Results", model_name)

    # Save the model state_dict (the recommended way to save in PyTorch)
    torch.save({
        'epoch': config['epochs'],
        'model_state_dict': model.state_dict(),
        'auc': auc,
        'input_dim': 320 # the official baseline uses 64*5
    }, full_save_path)

    print(f"💾 Weights saved to: {full_save_path}")
    return auc



In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# Create the folder and extract
if not os.path.exists(LOCAL_DATA_PATH):
    os.makedirs(LOCAL_DATA_PATH, exist_ok=True)
    print(f"📦 Extracting dataset to local disk {LOCAL_DATA_PATH}...")
    # -q = quiet mode, don't print thousands of file names
    !unzip -q {ZIP_PATH} -d {LOCAL_DATA_PATH}
    print("✅ Extraction complete!")
else:
    print("ℹ️ Dataset already exists, skipping extraction.")

# Skip training if model weights already exist

In [ ]:
# Force-create the directory (exist_ok=True: no error if it already exists)
if not os.path.exists(SAVE_DIR):
    try:
        os.makedirs(SAVE_DIR, exist_ok=True)
        print(f"✅ Created directory: {SAVE_DIR}")
    except Exception as e:
        print(f"❌ Could not create directory, check that Drive is mounted: {e}")

In [ ]:
# ==========================================
# Batch run over the four machine IDs
# ==========================================

# Make sure train_baseline and MIMII_Baseline_AE from above are already defined
baseline_results = {}

print("🚀 Starting official MIMII baseline evaluation...")
print("-" * 50)

for mid in SELECTED_IDS:
    target_path = os.path.join(BASE_PATH, mid)

    # Robustness check: if the path is wrong, search the data directory for this ID
    if not os.path.exists(target_path):
        print(f"⚠️ Path {target_path} does not exist, searching...")
        # The zip sometimes extracts with an extra nested directory; handle that automatically
        search_path = glob.glob(f"{LOCAL_DATA_PATH}/**/{mid}", recursive=True)
        if search_path:
            target_path = search_path[0]
        else:
            print(f"❌ No data found for {mid}, check the directory layout of the extracted zip.")
            continue

    print(f"\n[Processing] Machine ID: {mid}")
    try:
        # Train and evaluate (MY_CONFIG keeps the 50 epochs set above)
        auc_score = train_baseline(target_path, MY_CONFIG)
        baseline_results[mid] = auc_score
        print(f"✨ {mid} result: AUC = {auc_score:.4f}")
    except Exception as e:
        print(f"💥 {mid} run failed: {str(e)}")

# ==========================================
# Final report
# ==========================================
print("\n" + "="*50)
print("🏆 MIMII OFFICIAL BASELINE EXPERIMENT REPORT")
print("-" * 50)
print(f"{'Machine ID':<15} | {'Baseline AUC (0dB)':<15}")
print("-" * 50)
for mid in SELECTED_IDS:
    score = baseline_results.get(mid, "N/A")
    score_str = f"{score:.4f}" if isinstance(score, float) else score
    print(f"{mid:<15} | {score_str:<15}")
print("="*50)

In [ ]:
# Load weights
def load_trained_model(model_path, device):
    input_dim = MY_CONFIG['input_dim']
    model = MIMII_Baseline_AE(input_dim = input_dim).to(device)
    # Load weights
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    model.eval()
    return model

In [ ]:
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

# ==========================================
# 1. Core evaluation: per-file MSE score
# ==========================================
def get_eval_scores(model, eval_files, device):
    model.eval()
    y_pred_scores = []
    print(f"Computing reconstruction error for {len(eval_files)} evaluation clips...")

    with torch.no_grad():
        for file_name in tqdm(eval_files):
            # Official feature extraction: (frames_count, 320)
            data = file_to_vector_array(file_name)
            data_tensor = torch.FloatTensor(data).to(device)

            # Model reconstruction
            reconstructed = model(data_tensor)

            # Official logic: per-frame MSE, then the mean over all frames of the file
            mse_per_frame = torch.mean((data_tensor - reconstructed)**2, dim=1)
            file_error = torch.mean(mse_per_frame).cpu().item()
            y_pred_scores.append(file_error)

    return np.array(y_pred_scores)

# ==========================================
# 2. Threshold search: find the threshold that maximizes F1-score
# ==========================================
def find_best_threshold(y_true, y_scores):
    # Try 1000 candidate thresholds between the min and max score
    thresholds = np.linspace(min(y_scores), max(y_scores), 1000)
    best_f1 = 0
    best_threshold = 0
    best_metrics = {}

    for t in thresholds:
        # Decision: MSE > threshold -> abnormal (1)
        y_pred_binary = [1 if s > t else 0 for s in y_scores]

        # Compute precision, recall, F1
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred_binary, average='binary', zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = t
            best_metrics = {
                "threshold": t,
                "precision": precision,
                "recall": recall,
                "f1_score": f1,
                "accuracy": accuracy_score(y_true, y_pred_binary),
                "conf_matrix": confusion_matrix(y_true, y_pred_binary)
            }

    return best_metrics

# ==========================================
# 3. Main pipeline: classification check (e.g. id_04)
# ==========================================
def run_classification_pipeline(target_id):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_path = f"/content/drive/MyDrive/MIMII_Baseline_Results/baseline_fan_{target_id}.pth"

    # 1. Load the trained model
    print(f"--- Processing {target_id} ---")
    model = load_trained_model(model_path, device)

    # 2. Prepare evaluation files and labels (same split as in training)
    target_dir = os.path.join(BASE_PATH, target_id)
    normal_files = sorted(glob.glob(f"{target_dir}/normal/*.wav"))
    abnormal_files = sorted(glob.glob(f"{target_dir}/abnormal/*.wav"))

    # Evaluation set: as many normal clips as abnormal ones + all abnormal clips
    eval_files = normal_files[:len(abnormal_files)] + abnormal_files
    eval_labels = [0] * len(normal_files[:len(abnormal_files)]) + [1] * len(abnormal_files)

    # 3. MSE scores for all files
    scores = get_eval_scores(model, eval_files, device)

    # 4. Best classification metrics
    report = find_best_threshold(eval_labels, scores)

    # 5. Print results
    print("\n" + "="*45)
    print(f"📊 {target_id} final classification report (at best-F1 threshold)")
    print("-" * 45)
    print(f"Best threshold (MSE): {report['threshold']:.6f}")
    print(f"Accuracy:          {report['accuracy']:.4f}")
    print(f"Precision:         {report['precision']:.4f}")
    print(f"Recall:            {report['recall']:.4f}")
    print(f"F1-Score:          {report['f1_score']:.4f}")
    print("-" * 45)
    print(f"Confusion matrix:\n{report['conf_matrix']}")
    print("Note: [top-left: TN, top-right: FP, bottom-left: FN, bottom-right: TP]")
    print("="*45)

# Run the classification report
for id in SELECTED_IDS:
  run_classification_pipeline(id)